# Symbolic Fall Detector v3 (URFD, neck height in metres)

Reference: Applied Sciences 2026, 16(4), 1895, Eq. 1 and 7 to 10.

Open it from `notebooks/` (or the project root) and **Run All**. It needs `processed_data/kinect_height/` (made once with `python3 fall_detection_models/code/stgcn_urfd/kinect_neck_height.py`).

The model code lives in `fall_detection_models/code/stgcn_urfd/symbolic_v3.py`; this notebook only loads the data, fits each fold on its training videos and reports validation and test results on the same scene-disjoint folds as the ST-GCN.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd

PROJECT_ROOT = os.environ.get("PROJECT_ROOT") or ("." if os.path.isdir("splits") else "..")   # project root or notebooks/
sys.path.insert(0, os.path.join(PROJECT_ROOT, "fall_detection_models", "code", "stgcn_urfd"))
import symbolic_v3 as S

NPY = os.path.join(PROJECT_ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
KINECT = os.path.join(PROJECT_ROOT, "processed_data", "kinect_height")
split = json.load(open(os.path.join(PROJECT_ROOT, "splits", "split_scene_v2.json")))
cols = ["v", "f", "lab", "hw", "mm", "occ", "std", "hh", "H", "Dmm", "p40"]
urfd = pd.read_csv(os.path.join(PROJECT_ROOT, "URFD", "features", "urfall-cam0-falls.csv"), header=None, names=cols)

data = {v: (S.load_height(KINECT, v), np.load(os.path.join(NPY, f"{v}_frame_labels.npy")).astype(int))
        for v in sorted(split["group_of"])}
print(len(data), "videos")

70 videos


## Fit on train, score validation and test

In [2]:
def lying_heights(vids):
    out = []
    for v in vids:
        if v.startswith("fall"):
            h = np.nan_to_num(data[v][0], nan=2.0)
            i = urfd[(urfd.v == v) & (urfd.lab == 1)].f.values - 1
            out.append(h[i[i < len(h)]])
    return np.concatenate(out)


def report(rows):
    y = np.concatenate([r[1] for r in rows]); p = np.concatenate([r[2] for r in rows])
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    det = falls = clean = adl = eps = fr = 0
    for v, yy, pp in rows:                       # event definitions of the ST-GCN reports
        if v.startswith("fall"):
            falls += 1; det += int(pp[int(np.argmax(yy)):].any())
        else:
            adl += 1; clean += int(not pp.any()); fr += len(pp); eps += int(pp[0]) + int((np.diff(pp) == 1).sum())
    return dict(MCC=round(S.mcc(y, p), 3), F1=round(2 * pr * rc / max(pr + rc, 1e-9), 3), Precision=round(pr, 3),
                Recall=round(rc, 3), Specificity=round(tn / max(tn + fp, 1), 3), Accuracy=round((tp + tn) / len(y), 3),
                Falls=f"{det}/{falls}", ADL_no_alarm=f"{clean}/{adl}", Alarms_per_min=round(eps / (fr / S.FPS / 60), 2))


outputs = {}
for f in split["folds"]:
    m, train_mcc = S.fit([data[v] for v in f["train"]], lying_heights(f["train"]))
    print(f"fold {f['fold']}: tau_h={m.tau_h:.3f} m, tau_v={m.tau_v:.3f} m/s, tau_s={m.tau_s:.3f} m | train trigger MCC={train_mcc:.3f}")
    for part in ("val", "test"):
        for v in f[part]:
            outputs[(part, v)] = dict(fold=f["fold"], **m.predict(data[v][0]))

for part in ("val", "test"):
    for key in ("trigger", "state"):
        rows = [(v, data[v][1], o[key]) for (p, v), o in outputs.items() if p == part]
        print(f"{part:4s} {key:8s}", report(rows))

fold 0: tau_h=0.750 m, tau_v=0.411 m/s, tau_s=0.451 m | train trigger MCC=0.452


fold 1: tau_h=0.750 m, tau_v=0.411 m/s, tau_s=0.374 m | train trigger MCC=0.523
val  trigger  {'MCC': np.float64(0.214), 'F1': 0.264, 'Precision': 0.38, 'Recall': 0.203, 'Specificity': 0.958, 'Accuracy': 0.873, 'Falls': '6/6', 'ADL_no_alarm': '2/11', 'Alarms_per_min': 9.13}
val  state    {'MCC': np.float64(0.238), 'F1': 0.335, 'Precision': 0.248, 'Recall': 0.517, 'Specificity': 0.801, 'Accuracy': 0.769, 'Falls': '6/6', 'ADL_no_alarm': '2/11', 'Alarms_per_min': 10.27}
test trigger  {'MCC': np.float64(0.402), 'F1': 0.405, 'Precision': 0.735, 'Recall': 0.28, 'Specificity': 0.982, 'Accuracy': 0.876, 'Falls': '30/30', 'ADL_no_alarm': '23/40', 'Alarms_per_min': 5.03}
test state    {'MCC': np.float64(0.402), 'F1': 0.493, 'Precision': 0.487, 'Recall': 0.499, 'Specificity': 0.907, 'Accuracy': 0.845, 'Falls': '30/30', 'ADL_no_alarm': '24/40', 'Alarms_per_min': 5.03}


## Save the test-fold outputs for the fusion
Every URFD video is in exactly one test fold and is scored by the model whose training never saw its room. Per frame, in the order of `*_frame_labels.npy`.

In [3]:
save = {}
for (p, v), o in outputs.items():
    if p == "test":
        for key in ("trigger", "state", "support80", "h", "v"):
            save[f"{v}__{key}"] = o[key]
np.savez(os.path.join(PROJECT_ROOT, "symbolic_v3_predictions.npz"), **save)
print("saved symbolic_v3_predictions.npz")

saved symbolic_v3_predictions.npz
